# 04 — GenAI

## Task 4.1

In [ ]:
!pip install -q google-genai pydantic
import json, time
import pandas as pd
from google import genai
from google.genai import types

MODEL_NAME = "gemini-3-flash-preview"   # replace with the Flash model name from your Session 22.1 notebook
API_KEY = input("Paste your key: ")
client = genai.Client(api_key=API_KEY)

ALLOWED = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee", "illness_or_emergency", "work_or_schedule", "other"]
SCHEMA = {
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}
SYSTEM_TEXT = "You are a careful analyst for an outpatient clinic chain. You only answer in the requested JSON format."

api_calls = 0
last_raw = ""

def call_model(prompt):
    global api_calls
    api_calls += 1
    return client.models.generate_content(
        model=MODEL_NAME, contents=prompt,
        config=types.GenerateContentConfig(system_instruction=SYSTEM_TEXT, temperature=0,
            max_output_tokens=1000, response_mime_type="application/json", response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")))

def craft_prompt(note_text):
    return f"""Context: A clinic's call-centre phoned a patient who missed an appointment and wrote the note below.
Role: You are a clinic operations analyst.
Action: Classify why the patient missed the visit. root_cause must be exactly one of: {", ".join(ALLOWED)}.
If the note does not clearly give a reason, answer other instead of guessing.
Tone: Neutral and factual.
Format: JSON with root_cause, confidence (a number from 0 to 1), summary (at most 25 words), recommended_action.

Note: {note_text}"""

def analyse_note(note_text):
    global last_raw
    response = call_model(craft_prompt(note_text))
    last_raw = response.text
    return json.loads(last_raw)

notes = pd.read_csv("data/followup_notes.csv")
print(analyse_note(notes.loc[0, "note_text"]))

In [ ]:
# clear the cell output before saving, then search the notebook for the first characters of your key (Edit -> Find)

## Task 4.2

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def check_allowed(insight):
    if insight.root_cause not in ALLOWED:
        raise ValueError(f"root_cause {insight.root_cause!r} is not in the allowed list")

rows = []
for _, n in notes.head(10).iterrows():
    note_text = n["note_text"]
    if pd.isna(note_text) or str(note_text).strip() == "":
        rows.append({"note_id": n["note_id"], "appointment_id": n["appointment_id"], "root_cause": "other", "confidence": 0.0,
                     "summary": "No note recorded", "recommended_action": "Re-call patient",
                     "validation_status": "skipped_blank", "raw_response": ""})
        continue
    last_raw = ""
    data = {}
    try:
        data = analyse_note(note_text)
        check_allowed(FollowupInsight(**data))
        status = "valid"
    except Exception as e:
        status = "invalid"
        print(n["note_id"], "invalid:", e)
    if not isinstance(data, dict):
        data = {}
    rows.append({"note_id": n["note_id"], "appointment_id": n["appointment_id"],
                 "root_cause": data.get("root_cause", ""), "confidence": data.get("confidence", ""),
                 "summary": data.get("summary", ""), "recommended_action": data.get("recommended_action", ""),
                 "validation_status": status, "raw_response": last_raw})
    time.sleep(13)

analysed = pd.DataFrame(rows)
analysed.to_csv("notes_analysed.csv", index=False)
print(analysed[["note_id", "root_cause", "confidence", "validation_status"]])

## Task 4.3 

In [ ]:
bad_replies = {"confidence 1.7": {"root_cause": "other", "confidence": 1.7, "summary": "Unclear reason given", "recommended_action": "Re-call patient"},
               "missing recommended_action": {"root_cause": "other", "confidence": 0.5, "summary": "Unclear reason given"}}
for label, reply in bad_replies.items():
    try:
        FollowupInsight(**reply)
    except ValidationError as e:
        print(label, "->", e)

results = []
for nid in ["N007", "N009"]:
    text = notes.loc[notes["note_id"] == nid, "note_text"].iloc[0]
    for version, prompt in [("V1", f"Classify this note: {text}"), ("V2", craft_prompt(text))]:
        reply = json.loads(call_model(prompt).text)
        results.append({"note": nid, "prompt": version, "root_cause": reply.get("root_cause"),
                        "in_allowed_list": reply.get("root_cause") in ALLOWED})
        time.sleep(13)
print(pd.DataFrame(results))
print("Total API calls used in Tasks 4.1-4.3:", api_calls)

## Task 4.4

In [ ]:
appts = pd.read_csv("appointments_clean.csv")
merged = analysed.merge(appts, on="appointment_id")
print("Merged rows:", len(merged))
print(pd.crosstab(merged["root_cause"], merged["reminder_sent"]))